In [ ]:
# List of required packages
required_packages <- c( 
    "dplyr", 
    "tidyr",
    "data.table",
    "ggplot2",
    "lubridate",
    "broom",
    "scales",
    "patchwork",
    "glmnet"
)

# Function to check and install missing packages
install_and_load <- function(packages) {
  installed_packages <- rownames(installed.packages())
  for (pkg in packages) {
    if (!pkg %in% installed_packages) {
      install.packages(pkg, dependencies = TRUE)
    }
    suppressMessages(library(pkg, character.only = TRUE))
  }
}

# Install and load the required packages
install_and_load(required_packages)

In [ ]:
# Define the file name
file_names <- c(
    "ukb_covariates_formatted.csv.gz", 
    "ukb_gp_formatted_from_assessment.csv.gz",
    "ukb_acute_coronary_events_inpatient_plus_death.csv.gz",
    "ukb_pci_cabg_events.csv.gz"
)

# Check if the file exists in the current directory
for (file in file_names) {
    if (!file.exists(file)) {
      message("File not found. Downloading from DNAnexus...")

      # Execute the dx download command
      system(paste("dx download project-XXXXXXXXXXXXXXXXXXXXXXXX:/path/to/scratch/", file, sep = ""))
    }
}

# Load the data
cov <- fread("ukb_covariates_formatted.csv.gz")
gp  <- fread("ukb_gp_formatted_from_assessment.csv.gz")
gp_records <- fread("ukb_gp_formatted.csv.gz")
cad_events <- fread("ukb_acute_coronary_events_inpatient_plus_death.csv.gz")
cad_oper <- fread("ukb_pci_cabg_events.csv.gz")

data_raw <- merge(cov, gp, by = "eid", all = FALSE)

In [ ]:
# Restrict to Europeans for now
data_raw <- data_raw %>% filter(caucasian == 1)

In [ ]:
# Restrict to EIDs which are not part of the GP Clinical records 
# eid_remove <- unique(gp_records$eid)
# data_raw <- data_raw %>% filter(eid %notin% eid_remove)

In [ ]:
colnames(data_raw)

In [ ]:
data_raw[, proxy_birthdate := ymd(proxy_birthdate)]

In [ ]:
setDT(data_raw)  # re-establish by-reference data.table semantics
data_raw[, age_at_LDL0 := time_length(interval(proxy_birthdate, t_landmark), "years")]

In [ ]:
# Make interaction variables
data_raw$A_x_ldl_prs <- data_raw$A * data_raw$ldl_prs
data_raw$A_x_cad_prs <- data_raw$A * data_raw$cad_prs
data_raw$A_x_cvd_prs <- data_raw$A * data_raw$cvd_prs

In [ ]:
# Outcome and control variables
y_var   <- "LDL1"

log_vars    <- c("ldl_level", "triglyceride_level", "crp_level",
                 "hba1c_level", "alt_level", "hdl_level")  # log() in model
num_vars    <- c("townsend_index", "systolic_bp", "diastolic_bp", "bmi", 
                 "age_assessment", "ldl_prs", "n_rx_days_365d", "n_clin_days_365d", 
                 "body_fat_percentage", "apob_level", "urate_level")
factor_vars <- c("smoking_status", "alcohol_use", "sex", "prior_statin", "prior_CAD", 
                 "has_chest_pain", "has_diabetes", "edu_quals_cat")

In [ ]:
# Print NAs
vars <- c(log_vars, num_vars, factor_vars)

# NA counts
na_counts <- sapply(vars, function(v) {
  if (!v %in% names(data_raw)) return(NA_integer_)
  sum(is.na(data_raw[[v]]))
})

# Display nicely (sorted)
na_df <- data.frame(
  variable = names(na_counts),
  n_NA = as.integer(na_counts),
  pct_NA = round(100 * as.integer(na_counts) / nrow(data_raw), 2)
)

na_df <- na_df[order(-na_df$n_NA), ]
print(na_df, row.names = FALSE)

In [ ]:
# ---- helper: complete-case restriction respecting log() constraints ----
prep_complete_cases <- function(dt, y, log_vars, num_vars, factor_vars, log_y = FALSE) {
  vars_needed <- unique(c(y, log_vars, num_vars, factor_vars))
  keep <- stats::complete.cases(dt[, ..vars_needed])

  # log() covariates require strictly positive values
  for (v in log_vars) keep <- keep & is.finite(dt[[v]]) & dt[[v]] > 0

  # outcome constraints depend on whether it's logged
  keep <- keep & is.finite(dt[[y]])
  if (isTRUE(log_y)) keep <- keep & dt[[y]] > 0

  dt[keep]
}

data <- prep_complete_cases(data_raw, y_var, log_vars, num_vars, factor_vars, log_y = TRUE)

In [ ]:
# ---- build formula programmatically ----
log_terms    <- sprintf("I(log(%s))", log_vars)
num_terms    <- num_vars
factor_terms <- sprintf("factor(%s)", factor_vars)

In [ ]:
# Not adjusting for LDL0
fit_unadj <- lm(I(log(LDL1)) ~ age_assessment + sex + A + ldl_prs + A_x_ldl_prs, data = data)
summary(fit_unadj)

In [ ]:
# Adjusting for LDL0
rhs_terms <- c(log_terms, num_terms, factor_terms, "A + ldl_prs + A_x_ldl_prs")
form <- as.formula(paste0("I(log(", y_var, ")) ~ ", paste(rhs_terms, collapse = " + ")))

fit_adj <- lm(form, data = data)
summary(fit_adj)

In [ ]:
# Ensure binary A
data <- data %>%
  mutate(
    A = as.integer(A),
    M = as.integer(has_LDL1 == TRUE)
  )
stopifnot(all(data$A %in% c(0L, 1L)))

# ----------------------------
# Denominator model: P(A=1 | X) via cv.glmnet
# ----------------------------
rhs_terms <- c(log_terms, num_terms, factor_terms, "ldl_prs", "cad_prs")
form <- as.formula(paste0("A ~ ", paste(rhs_terms, collapse = " + ")))

# Build model matrix for glmnet; drop intercept (glmnet adds it by default)
mm <- model.matrix(form, data = data)
y  <- model.response(model.frame(form, data = data))

x <- mm[, colnames(mm) != "(Intercept)", drop = FALSE]

# Fit cross-validated penalised logistic regression
mA_den <- cv.glmnet(
  x = x, y = y,
  family = "binomial",
  type.measure = "deviance",
  alpha = 1
)
coef(mA_den)

# Predict probabilities at lambda.min (or lambda.1se for more shrinkage)
pA1_den <- as.numeric(predict(mA_den, newx = x, s = "lambda.min", type = "response"))
pA1_den <- pmin(pmax(pA1_den, 1e-6), 1 - 1e-6)  # avoid 0/1

pA0_den <- 1 - pA1_den

# ----------------------------
# Stabilised weights
# ----------------------------
pA1_num <- mean(data$A == 1L, na.rm = TRUE)
pA0_num <- 1 - pA1_num

wA <- ifelse(data$A == 1L, pA1_num / pA1_den, pA0_num / pA0_den)

# Truncate weights
q05 <- quantile(wA, 0.05, na.rm = TRUE)
q95 <- quantile(wA, 0.95, na.rm = TRUE)
wA  <- pmin(pmax(wA, q05), q95)

# Attach to data
data <- data %>%
  mutate(
    pA1_den = pA1_den,
    pA_den  = ifelse(A == 1L, pA1_den, pA0_den),
    wA_ipw  = wA
  )

# Diagnostics
summary(data$wA_ipw)
quantile(data$wA_ipw, probs = c(0, 0.5, 0.9, 0.95, 0.99, 1), na.rm = TRUE)

# ----------------------------
# Weighted outcome model
# ----------------------------
fit_msm <- lm(
  I(log(LDL1)) ~ age_assessment + sex + A + ldl_prs + A_x_ldl_prs,
  data = data,
  weights = wA_ipw
)
summary(fit_msm)

In [ ]:
pal <- c(
  "Unadjusted" = "#7570B3",
  "Adjusted for baseline LDL" = "#D95F02",
  "MSM/IPTW" = "#1B9E77"
)

theme_pub <- theme_minimal(base_size = 12) +
  theme(
    panel.grid.minor = element_blank(),
    panel.grid.major.y = element_blank(),
    legend.title = element_blank(),
    legend.position = "top",
    plot.title.position = "plot",
    plot.title = element_text(face = "bold"),
    plot.subtitle = element_text(colour = "grey25"),
    axis.title = element_text(face = "bold")
  )

# ----------------------------
# Helper: tidy models, exponentiate selected terms for interpretability
# For log outcome:
#   exp(beta_A) is the multiplicative effect on LDL1 for A=1 vs A=0 at PRS=0
#   exp(beta_A:PRS) is multiplicative change in treatment effect per 1 SD PRS
# ----------------------------
tidy_model <- function(fit, model_label) {
  broom::tidy(fit, conf.int = TRUE) %>%
    mutate(model = model_label)
}

td_unadj <- tidy_model(fit_unadj, "Unadjusted")
td_adj   <- tidy_model(fit_adj,   "Adjusted for baseline LDL")
td_msm   <- tidy_model(fit_msm,   "MSM/IPTW")

td_all <- bind_rows(td_unadj, td_adj, td_msm) %>%
  mutate(
    term_pretty = recode(term,
      "(Intercept)" = "Intercept",
      "age_at_LDL0"  = "Age at LDL0 (years)",
      "sex"          = "Sex",
      "A"            = "Statin initiation (A)",
      "ldl_prs"      = "LDL PRS (G)",
      "A_x_ldl_prs"    = "Interaction (A × G)",
      "I(log(LDL0))" = "Baseline log(LDL0)"
    ),
    # exponentiated effects (ratio scale) for key terms only
    exp_estimate = exp(estimate),
    exp_conf_low = exp(conf.low),
    exp_conf_high = exp(conf.high)
  )

In [ ]:
focus_terms <- c("Statin initiation (A)", "Interaction (A × G)")

df_focus <- td_all %>%
  filter(term_pretty %in% focus_terms) %>%
  mutate(
    model = factor(model, levels = names(pal)),
    # Rename facets into plain English
    term_pretty = recode(term_pretty,
                         "Statin initiation (A)" = "Change in LDL after starting statins",
                         "Interaction (A × G)"   = "Modification of statin response by LDL PRS") %>%
      factor(levels = c("Change in LDL after starting statins",
                        "Modification of statin response by LDL PRS"))
  )

p_forest_focus <- ggplot(
  df_focus,
  aes(
    y = model,
    x = exp_estimate,
    xmin = exp_conf_low,
    xmax = exp_conf_high,
    colour = model
  )
) +
  # Reference: no difference between groups
  geom_vline(xintercept = 1, linewidth = 0.6, linetype = 2, colour = "grey45") +
  geom_errorbarh(height = 0.20, linewidth = 0.9) +
  geom_point(size = 3.0) +
  facet_wrap(~ term_pretty, ncol = 1, scales = "free_y") +
  scale_colour_manual(values = pal) +
  # Keep log axis but label it in intuitive terms
  scale_x_log10(
    breaks = c(0.5, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1, 1.25, 1.5, 2),
    labels = function(x) paste0(round((x - 1) * 100), "%")
  ) +
  labs(
    # title = "Statins lower LDL, but LDL PRS does not change the size of this effect",
    # subtitle = "Values left of 0% indicate lower LDL after starting statins. 0% indicates no change.",
    x = "Change in LDL (percentage change)",
    y = NULL
  ) +
  theme_pub +
  theme(
    legend.position = "top",
    strip.text = element_text(face = "bold")
  )
p_forest_focus
ggsave(
  plot = p_forest_focus,
  width = 8,
  height = 6,
  dpi = 300,     
  filename = "statin_ldl_interaction_forest_from_assessment.png"
)

In [ ]:
stopifnot(all(c("wA_ipw", "pA1_den", "A") %in% names(data)))

# Colour scheme (Okabe–Ito) consistent with earlier figures
pal_A <- c("No statin initiation" = "#0072B2", "Initiated statin" = "#D55E00")

# 1) Histogram of fitted initiation probabilities Pr(A=1 | X)
#    Show distributions by observed initiation status for interpretability.
df_p <- data %>%
  filter(is.finite(pA1_den), pA1_den > 0, pA1_den < 1, !is.na(A)) %>%
  mutate(A_f = factor(A, levels = c(0, 1), labels = c("No statin initiation", "Initiated statin")))

df_cal <- df_p %>%
  mutate(bin = ntile(pA1_den, 20)) %>%
  group_by(bin) %>%
  summarise(
    p_mean = mean(pA1_den),
    A_rate = mean(A),
    n = n(),
    .groups = "drop"
  )

# Common x scale to control tick density
scale_x_pct <- scale_x_continuous(
  labels = percent_format(accuracy = 1),
  breaks = seq(0, 1, by = 0.05),
  expand = expansion(mult = c(0.01, 0.01))
)

p_p_hist <- ggplot(df_p, aes(x = pA1_den, fill = A_f)) +
  geom_histogram(bins = 50, alpha = 0.55, colour = "white", linewidth = 0.2) +
  geom_density(aes(colour = A_f), linewidth = 0.9, alpha = 0.0) +
  scale_fill_manual(values = pal_A) +
  scale_colour_manual(values = pal_A) +
  scale_x_pct +
  labs(
    title = "Predicted probability\nof starting statins",
    x = "Predicted probability\nof statin initiation",
    y = "Number of participants"
  ) +
  theme_pub +
  theme(
    axis.title.x = element_text(margin = margin(t = 6)),
    axis.text.x = element_text(angle = 0)
  )

p_cal <- ggplot(df_cal, aes(x = p_mean, y = A_rate)) +
  geom_abline(slope = 1, intercept = 0, linetype = 2, colour = "grey45", linewidth = 0.7) +
  geom_point(size = 3.0, alpha = 0.95, colour = "grey15") +
  geom_line(linewidth = 0.9, alpha = 0.9, colour = "grey15") +
  scale_x_pct +
  scale_y_continuous(
    labels = percent_format(accuracy = 1),
    breaks = seq(0, 1, by = 0.05)
  ) +
  labs(
    title = "Statin initiation\nmodel check",
    x = "Predicted probability\nof statin initiation",
    y = "Observed probability\nof statin initiation"
  ) +
  theme_pub +
  theme(
    axis.title.x = element_text(margin = margin(t = 6))
  )

# Give panels more breathing room
p_ipw_diag <- p_p_hist + p_cal +
  patchwork::plot_layout(ncol = 2) &
  theme(plot.margin = margin(10, 12, 10, 12))

p_ipw_diag
ggsave(
  plot = p_ipw_diag,
  width = 8,
  height = 6,
  dpi = 300,
  filename = "iptw_diagnostics_plot_from_assessment.png"
)

In [ ]:
# Ensure data.table
dt <- as.data.table(copy(data_raw))
ev <- as.data.table(copy(cad_events))

# Types
dt[, t_landmark := as.IDate(t_landmark)]
ev[, event_date := as.IDate(event_date)]

# 10-year window end (use lubridate to handle leap years cleanly)
dt[, cad_end := as.IDate(t_landmark %m+% years(10))]

# Keep only events on/after landmark and within 10y
cand <- ev[dt, on = .(eid), allow.cartesian = TRUE][
  event_date > t_landmark & event_date <= cad_end,
  .(eid, CAD1_dt = event_date)
]

# Pick earliest event per person
if (nrow(cand) == 0L) {
  dt[, `:=`(CAD1_dt = as.IDate(NA), CAD1 = 0L)]
} else {
  setorder(cand, eid, CAD1_dt)
  cad_pick <- cand[, .SD[1L], by = eid]

  dt[cad_pick, CAD1_dt := i.CAD1_dt, on = "eid"]
  dt[, CAD1 := as.integer(!is.na(CAD1_dt))]
}

# Drop helper
dt[, cad_end := NULL]

print(sum(dt$CAD1))

In [ ]:
y_var <- "CAD1"

data <- prep_complete_cases(dt, y_var, log_vars, num_vars, factor_vars)

In [ ]:
# Not adjusting for LDL0
fit_unadj <- glm(
    CAD1 ~ age_assessment + sex + A + cad_prs + A_x_cad_prs, 
    family = "binomial",
    data = data
)
summary(fit_unadj)

In [ ]:
# Adjusting for LDL0
rhs_terms <- c(log_terms, num_terms, factor_terms, "A + cad_prs + A_x_cad_prs")
form <- as.formula(paste0("CAD1 ~ ", paste(rhs_terms, collapse = " + ")))

fit_adj <- glm(form, family = "binomial", data = data)
summary(fit_adj)

In [ ]:
# Ensure binary A
data <- data %>%
  mutate(
    A = as.integer(A),
    M = as.integer(has_LDL1 == TRUE)
  )
stopifnot(all(data$A %in% c(0L, 1L)))

# ----------------------------
# Denominator model: P(A=1 | X) via cv.glmnet
# ----------------------------
rhs_terms <- c(log_terms, num_terms, factor_terms, "ldl_prs", "cad_prs")
form <- as.formula(paste0("A ~ ", paste(rhs_terms, collapse = " + ")))

# Build model matrix for glmnet; drop intercept (glmnet adds it by default)
mm <- model.matrix(form, data = data)
y  <- model.response(model.frame(form, data = data))

x <- mm[, colnames(mm) != "(Intercept)", drop = FALSE]

# Fit cross-validated penalised logistic regression
mA_den <- cv.glmnet(
  x = x, y = y,
  family = "binomial",
  type.measure = "deviance",
  alpha = 1
)
coef(mA_den)

# Predict probabilities at lambda.min (or lambda.1se for more shrinkage)
pA1_den <- as.numeric(predict(mA_den, newx = x, s = "lambda.min", type = "response"))
pA1_den <- pmin(pmax(pA1_den, 1e-6), 1 - 1e-6)  # avoid 0/1

pA0_den <- 1 - pA1_den

# ----------------------------
# Stabilised weights
# ----------------------------
pA1_num <- mean(data$A == 1L, na.rm = TRUE)
pA0_num <- 1 - pA1_num

wA <- ifelse(data$A == 1L, pA1_num / pA1_den, pA0_num / pA0_den)

# Truncate weights
q05 <- quantile(wA, 0.05, na.rm = TRUE)
q95 <- quantile(wA, 0.95, na.rm = TRUE)
wA  <- pmin(pmax(wA, q05), q95)

# Attach to data
data <- data %>%
  mutate(
    pA1_den = pA1_den,
    pA_den  = ifelse(A == 1L, pA1_den, pA0_den),
    wA_ipw  = wA
  )

# Diagnostics
summary(data$wA_ipw)
quantile(data$wA_ipw, probs = c(0, 0.5, 0.9, 0.95, 0.99, 1), na.rm = TRUE)

# ----------------------------
# Use in MSM-style weighted outcome model (no LDL0 in outcome model)
# NOTE: use weights= (plural)
# ----------------------------
fit_msm <- glm(CAD1 ~ age_assessment + sex + A + cad_prs + A_x_cad_prs,
              data = data,
              family = quasibinomial(),
              weights = wA_ipw)

summary(fit_msm)

In [ ]:
td_all <- bind_rows(
  tidy_model(fit_unadj, "Unadjusted"),
  tidy_model(fit_adj,   "Adjusted for baseline LDL"),
  tidy_model(fit_msm,   "MSM/IPTW")
) %>%
  mutate(
    term_pretty = recode(term,
      "(Intercept)" = "Intercept",
      "age_at_LDL0"  = "Age at LDL0 (years)",
      "sex"          = "Sex",
      "A"            = "Statin initiation (A)",
      "cad_prs"      = "CAD PRS (G)",
      "A_x_cad_prs"    = "Interaction (A × G)",
      "I(log(LDL0))" = "Baseline log(LDL0)"
    ),
    # logistic regression: exp(beta) is odds ratio
    exp_estimate  = exp(estimate),
    exp_conf_low  = exp(conf.low),
    exp_conf_high = exp(conf.high)
  )

focus_terms <- c("Statin initiation (A)", "Interaction (A × G)")

df_focus <- td_all %>%
  filter(term_pretty %in% focus_terms) %>%
  mutate(
    model = factor(model, levels = names(pal)),
    term_pretty = recode(term_pretty,
      "Statin initiation (A)" = "Association of statin initiation with AMI (odds ratio)",
      "Interaction (A × G)"   = "Effect modification by CAD PRS (OR per 1 SD)"
    ) %>%
      factor(levels = c(
        "Association of statin initiation with AMI (odds ratio)",
        "Effect modification by CAD PRS (OR per 1 SD)"
      ))
  )

p_forest_focus <- ggplot(
  df_focus,
  aes(
    y = model,
    x = exp_estimate,
    xmin = exp_conf_low,
    xmax = exp_conf_high,
    colour = model
  )
) +
  geom_vline(xintercept = 1, linewidth = 0.6, linetype = 2, colour = "grey45") +
  geom_errorbarh(height = 0.20, linewidth = 0.9) +
  geom_point(size = 3.0) +
  facet_wrap(~ term_pretty, ncol = 1, scales = "free_y") +
  scale_colour_manual(values = pal) +
  scale_x_log10(
    breaks = c(0.5, 0.6, 0.7, 0.8, 0.9, 1, 1.1, 1.2, 1.3),
    labels = scales::label_number(accuracy = 0.01)
  ) +
  labs(
    x = "Odds ratio (log scale)",
    y = NULL
  ) +
  theme_pub +
  theme(
    legend.position = "top",
    strip.text = element_text(face = "bold")
  )

p_forest_focus
ggsave(
  plot = p_forest_focus,
  width = 8,
  height = 6,
  dpi = 300,     
  filename = "statin_ami_interaction_forest_from_assessment.png"
)